# Phase 4: Embedding & Vector Search Optimization Experiments

**Project:** LabelLens (Nutrition & Food Label Analyzer)  
**Dataset:** Clean Open Food Facts subset (`data/clean/products.parquet`, ~196k products)  
**Vector Store:** Qdrant (Named vectors: `text_dense`, `image_clip`, `bm25`)

---

## 1. Experimental Methodology

To objectively compare vector models, query representations, HNSW graph parameters, and quantization strategies before establishing our Phase 5 hybrid retrieval golden set, we execute a **proxy retrieval test** (`scripts/mini_eval.py`):
- **Sample:** 300 representative products sampled from `data/clean/products.parquet`.
- **Query Formulation:** Natural user query proxy formed by combining the product's brand and name (e.g., `"Marks & Spencer British Natural Goat's Cheese"`).
- **Metrics Recorded:**
  - **Recall@1:** Fraction of queries where the exact target product was ranked #1.
  - **Recall@5:** Fraction of queries where the target appeared in the top 5 results.
  - **Recall@10:** Fraction of queries where the target appeared in the top 10 results.
  - **MRR (Mean Reciprocal Rank):** Average reciprocal rank $\frac{1}{\text{rank}}$ across all queries.
  - **Average Latency:** Wall-clock time per query in milliseconds (including text embedding and vector search).

In [1]:
import json
from pathlib import Path

import pandas as pd

# Load recorded experiment benchmark results
results_file = Path("experiment_results.json")
if results_file.exists():
    with open(results_file, encoding="utf-8") as f:
        raw_data = json.load(f)
else:
    raw_data = {}

print(f"Loaded {len(raw_data)} experimental configurations.")

Loaded 9 experimental configurations.


## 2. Benchmark Summary Table

| Experiment | Configuration / Setting | Recall@1 | Recall@5 | Recall@10 | MRR | Latency (ms) | Notes |
|---|---|:---:|:---:|:---:|:---:|:---:|---|
| **Baseline** | `bge-small` (384-d), prefix=True, ef=None, quant=off | **0.807** | **0.983** | **0.983** | **0.882** | 27.3 ms | Default configuration |
| **Exp A: Prefix** | with instruction prefix | 0.807 | 0.983 | 0.983 | 0.882 | 27.3 ms | BGE query instruction prefix enabled |
| | without instruction prefix | 0.787 | 0.970 | 0.983 | 0.866 | 26.0 ms | Drop of -2.0% Recall@1, -1.6% MRR |
| **Exp B: Model Size** | `bge-small-en-v1.5` (384-d) | 0.807 | 0.983 | 0.983 | 0.882 | 27.3 ms | Lightweight, fast CPU inference |
| | `bge-base-en-v1.5` (768-d) | **0.983** | **1.000** | **1.000** | **0.991** | 24.1 ms | Major accuracy leap (+17.6% Recall@1) |
| **Exp C: HNSW ef** | `ef = 16` | 0.773 | 0.933 | 0.937 | 0.842 | 25.2 ms | Explores fewer neighbors; under-retrieves |
| | `ef = 64` | 0.810 | 0.983 | 0.983 | 0.884 | 25.0 ms | Sweet spot balancing accuracy & speed |
| | `ef = 256` | 0.817 | 0.990 | 0.990 | 0.891 | 28.1 ms | Maximum recall, slight latency overhead |
| **Exp D: Quantization** | FP32 (quantization OFF) | 0.807 | 0.983 | 0.983 | 0.882 | 27.3 ms | Baseline memory footprint (~4 bytes/dim) |
| | INT8 Scalar Quantization (rescore=False) | 0.807 | 0.983 | 0.983 | 0.882 | 31.5 ms | **75% RAM reduction** with 0% recall loss |
| | INT8 Scalar Quantization (rescore=True) | 0.807 | 0.983 | 0.983 | 0.883 | 30.8 ms | Exact vector re-ranking on top candidates |

In [2]:
# Display structured summary in DataFrame format
table_rows = [
    {
        "Experiment": "Baseline",
        "Setting": "bge-small + prefix (default)",
        "Recall@1": 0.8067,
        "Recall@5": 0.9833,
        "MRR": 0.8824,
        "Latency (ms)": 27.33,
        "Decision": "Reference baseline",
    },
    {
        "Experiment": "A: Query Prefix",
        "Setting": "Prefix: Yes",
        "Recall@1": 0.8067,
        "Recall@5": 0.9833,
        "MRR": 0.8824,
        "Latency (ms)": 27.33,
        "Decision": "Keep (Higher Recall)",
    },
    {
        "Experiment": "A: Query Prefix",
        "Setting": "Prefix: No",
        "Recall@1": 0.7867,
        "Recall@5": 0.9700,
        "MRR": 0.8662,
        "Latency (ms)": 26.04,
        "Decision": "Discard (-2% Recall@1)",
    },
    {
        "Experiment": "B: Model Size",
        "Setting": "bge-small (384-d)",
        "Recall@1": 0.8067,
        "Recall@5": 0.9833,
        "MRR": 0.8824,
        "Latency (ms)": 27.33,
        "Decision": "Low-resource option",
    },
    {
        "Experiment": "B: Model Size",
        "Setting": "bge-base (768-d)",
        "Recall@1": 0.9833,
        "Recall@5": 1.0000,
        "MRR": 0.9906,
        "Latency (ms)": 24.05,
        "Decision": "Keep for high-accuracy tier",
    },
    {
        "Experiment": "C: HNSW ef",
        "Setting": "ef = 16",
        "Recall@1": 0.7733,
        "Recall@5": 0.9333,
        "MRR": 0.8423,
        "Latency (ms)": 25.16,
        "Decision": "Discard (Insufficient Recall)",
    },
    {
        "Experiment": "C: HNSW ef",
        "Setting": "ef = 64",
        "Recall@1": 0.8100,
        "Recall@5": 0.9833,
        "MRR": 0.8841,
        "Latency (ms)": 24.98,
        "Decision": "Keep as default query ef",
    },
    {
        "Experiment": "C: HNSW ef",
        "Setting": "ef = 256",
        "Recall@1": 0.8167,
        "Recall@5": 0.9900,
        "MRR": 0.8908,
        "Latency (ms)": 28.13,
        "Decision": "Optional deep search",
    },
    {
        "Experiment": "D: Quantization",
        "Setting": "FP32 (Quantization Off)",
        "Recall@1": 0.8067,
        "Recall@5": 0.9833,
        "MRR": 0.8824,
        "Latency (ms)": 27.33,
        "Decision": "High RAM consumption",
    },
    {
        "Experiment": "D: Quantization",
        "Setting": "INT8 (rescore=False)",
        "Recall@1": 0.8067,
        "Recall@5": 0.9833,
        "MRR": 0.8824,
        "Latency (ms)": 31.46,
        "Decision": "Viable 75% RAM savings",
    },
    {
        "Experiment": "D: Quantization",
        "Setting": "INT8 (rescore=True)",
        "Recall@1": 0.8067,
        "Recall@5": 0.9833,
        "MRR": 0.8827,
        "Latency (ms)": 30.82,
        "Decision": "Keep permanently in production",
    },
]
df_experiments = pd.DataFrame(table_rows)
df_experiments

,Experiment,Setting,Recall@1,Recall@5,MRR,Latency (ms),Decision
0,Baseline,bge-small + prefix (default),0.8067,0.9833,0.8824,27.33,Reference baseline
1,A: Query Prefix,Prefix: Yes,0.8067,0.9833,0.8824,27.33,Keep (Higher Recall)
2,A: Query Prefix,Prefix: No,0.7867,0.9700,0.8662,26.04,Discard (-2% Recall@1)
3,B: Model Size,bge-small (384-d),0.8067,0.9833,0.8824,27.33,Low-resource option
4,B: Model Size,bge-base (768-d),0.9833,1.0000,0.9906,24.05,Keep for high-accuracy tier
5,C: HNSW ef,ef = 16,0.7733,0.9333,0.8423,25.16,Discard (Insufficient Recall)
6,C: HNSW ef,ef = 64,0.8100,0.9833,0.8841,24.98,Keep as default query ef
7,C: HNSW ef,ef = 256,0.8167,0.9900,0.8908,28.13,Optional deep search
8,D: Quantization,FP32 (Quantization Off),0.8067,0.9833,0.8824,27.33,High RAM consumption
9,D: Quantization,INT8 (rescore=False),0.8067,0.9833,0.8824,31.46,Viable 75% RAM savings


## 3. Analysis & Key Takeaways

### Experiment A: Query Instruction Prefix
BGE models are trained with asymmetric contrastive learning: documents are encoded as bare passages, while queries are instructed with `"Represent this sentence for searching relevant passages: "`. Omitting this prefix caused Recall@1 to degrade from **80.7%** down to **78.7%** (-2.0%) and MRR to drop from **0.882** to **0.866**, with essentially identical latency (27.3 ms vs 26.0 ms). This confirms that asymmetric query conditioning helps bridge the semantic gap between short user search queries (e.g. brand + product name) and dense product index cards. **We will keep the query instruction prefix enabled for all retrieval operations.**

### Experiment B: Model Size (`bge-small` vs `bge-base`)
Moving from `bge-small-en-v1.5` (384-d) to `bge-base-en-v1.5` (768-d) yielded a dramatic increase in retrieval power: Recall@1 surged from **80.7%** to **98.3%** (+17.6%), and MRR jumped from **0.882** to **0.991** with 100% Recall@5. The higher-dimensional latent space captures product brand nuances, multi-token qualifiers, and ingredient contexts far more effectively. While `bge-base` doubles the vector dimensionality (and thus the memory footprint of raw embeddings), search latency remained under 25 ms. For production, `bge-base` is the clear quality winner; however, `bge-small` remains an excellent low-resource option for local edge or mobile environments.

### Experiment C: HNSW Graph Exploration (`ef` Tuning)
Testing search-time `ef` values (16, 64, 256) demonstrated the fundamental HNSW trade-off between candidate graph exploration and accuracy. At `ef=16`, the search terminates too early in the proximity graph, resulting in an unacceptable drop in Recall@1 to **77.3%** (-3.4%) and Recall@5 to **93.3%** (-5.0%). Increasing `ef` to 64 restored Recall@1 to **81.0%** and Recall@5 to **98.3%** with virtually zero latency penalty (25.0 ms). Scaling up to `ef=256` reached **81.7%** Recall@1 and **99.0%** Recall@5 at 28.1 ms. **We will keep `ef=64` as our default query setting**, as it delivers 99% of maximum recall while maintaining minimal query overhead.

### Experiment D: Scalar Quantization (INT8 & Rescore)
Enabling INT8 scalar quantization with quantile 0.99 compresses each 32-bit float dimension down to an 8-bit integer, yielding a theoretical ~75% reduction in RAM usage across our ~388k vectors. Crucially, the quantization caused zero degradation in retrieval accuracy: Recall@1 remained identical at **80.7%** and Recall@5 at **98.3%**. Rescoring the top-10 candidates with the original non-quantized vectors produced a marginal improvement in MRR (**0.8827** vs **0.8824**) while maintaining ~30 ms average latency. Given that vector memory is the primary cost bottleneck on free cloud tiers and developer laptops, **INT8 scalar quantization with rescoring will be permanently kept in production**.

### Experiment E: Image Retrieval & Multimodal Search Reflection
With OpenCLIP ViT-B/32, packaging images and text descriptions share a 512-dimensional vector space. As verified with `try_image_search.py`, text queries like `"a bottle of ketchup"` or `"a jar of chocolate spread"` directly surface the correct visual packages, and attaching image vectors across downloaded shards flipped the `has_image` payload flags. This multimodal bridge sets up the hybrid retrieval and reranking layer in Phase 5.